# Explorando o pipeline — evasão escolar (INEP 2023)

Notebook de apoio à PoC. Serve para **olhar os dados em cada camada** sem sair do VS Code.

Ordem de uso:

1. rode `python -m src.pipeline` (cria o Bronze em **Delta Lake**) — depois use as seções 1 a 4;
2. rode `dbt build` dentro de `dbt/` (cria Silver e Gold) — depois use as seções 5 e 6.

> Requer o kernel do ambiente do projeto (`.venv` ou `poc`). No VS Code: canto superior direito → *Select Kernel*.

> ⚠ O Bronze é **Delta Lake** (uma pasta com `_delta_log/`, não um `.parquet` solto). Por isso, nas células do Bronze, lemos com `delta_scan('data/bronze/<tabela>')` — a função dedicada do DuckDB — e não com um caminho de arquivo.

## 0. Preparação

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# a raiz do projeto (este notebook está em notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Raiz do projeto:", RAIZ)

con = duckdb.connect()          # DuckDB em memória: só para consultar arquivos
con.execute(f"set file_search_path='{RAIZ}'")
# extensão delta: permite LER o Bronze (tabelas Delta) via delta_scan()
con.execute("install delta; load delta;")

# Por padrão o pandas TRUNCA a exibição. Aqui pedimos mais linhas/colunas.
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


def q(sql):
    """Executa SQL e devolve um DataFrame (renderiza bonito no notebook)."""
    return con.execute(sql).df()


def bronze(tabela):
    """Caminho absoluto da tabela Delta do Bronze, para usar em delta_scan()."""
    return (RAIZ / "data" / "bronze" / tabela).as_posix()

## 1. As fontes: o que a origem (INEP) nos entregou

Antes de qualquer transformação. Repare que **tudo é texto** aqui — CSV não guarda tipo, e algumas taxas vieram com **vírgula** decimal.

In [ ]:
q("select * from 'data/raw/taxas_municipios.csv' limit 10")

In [ ]:
# o JSON de UFs (sigla -> região)
q("select * from 'data/raw/ufs.json'")

## 2. O Bronze: o que o pipeline capturou (em Delta Lake)

Mesmo conteúdo das fontes, agora em Delta, com o metadado de ingestão (`_ingerido_em`).

In [ ]:
q(f"select * from delta_scan('{bronze('taxas')}') limit 10")

### 2.1 Por que Delta Lake? Olhe o histórico de versões

O Bronze é a camada de **preservação do bruto**. Delta acrescenta ao Parquet um **log de
transações**: cada carga da origem vira uma **versão imutável**. É isso que dá time
travel, auditoria e reprocessamento seguro.

> Rode `python scripts/time_travel.py` para criar a v1 (corrigindo a taxa > 100) e
> comparar as versões. A célula abaixo mostra o histórico atual.

In [ ]:
from deltalake import DeltaTable

dt = DeltaTable(bronze("taxas"))
print("Versão atual:", dt.version())
hist = pd.DataFrame(dt.history())[["version", "operation", "timestamp"]]
hist

In [ ]:
# tamanho em disco: CSV (origem) x Parquet do Delta (Bronze)
import os

csv = (RAIZ / "data/raw/taxas_municipios.csv").stat().st_size
pasta = RAIZ / "data/bronze/taxas"
pq = sum(f.stat().st_size for f in pasta.glob("*.parquet"))
print(f"CSV origem:      {csv:>9,} bytes")
print(f"Parquet (Delta): {pq:>9,} bytes")
print("\nO Parquet é colunar e comprimido: colunas com poucos valores distintos")
print("(uf, ano) comprimem muito. A vantagem cresce com a escala.")

## 3. Caça aos problemas de qualidade

O Bronze preserva o que chegou — **inclusive os defeitos**. Encontre-os aqui antes de
decidir como tratá-los na Silver. (Lista completa em `docs/anomalias-das-fontes.md`.)

In [ ]:
# a) taxas com VÍRGULA decimal em vez de ponto? (DEFEITO 1)
q(f"""
    select codigo_municipio, nome_municipio, taxa_abandono_fund
    from delta_scan('{bronze('taxas')}')
    where taxa_abandono_fund like '%,%'
""")

In [ ]:
# b) linhas integralmente duplicadas? (DEFEITO 6)
q(f"""
    select codigo_municipio, count(*) as vezes
    from delta_scan('{bronze('taxas')}')
    where codigo_municipio <> ''
    group by 1
    having count(*) > 1
""")

In [ ]:
# c) código de município ausente? (DEFEITO 5)
q(f"""
    select *
    from delta_scan('{bronze('taxas')}')
    where codigo_municipio is null or codigo_municipio = ''
""")

In [ ]:
# d) taxa de abandono IMPLAUSÍVEL (> 100)? (DEFEITO 4)
q(f"""
    select codigo_municipio, nome_municipio, uf, taxa_abandono_fund
    from delta_scan('{bronze('taxas')}')
    where try_cast(replace(taxa_abandono_fund, ',', '.') as double) > 100
""")

In [ ]:
# e) UF com grafia inconsistente (minúscula, espaços)? (DEFEITO 2)
q(f"""
    select distinct uf, '[' || uf || ']' as com_delimitador
    from delta_scan('{bronze('taxas')}')
    where uf <> upper(trim(uf))
""")

In [ ]:
# f) integridade: existe município apontando para UF inexistente no cadastro? (DEFEITO 3)
q(f"""
    select t.codigo_municipio, t.nome_municipio, t.uf
    from delta_scan('{bronze('taxas')}') t
    left join delta_scan('{bronze('ufs')}') u
           on upper(trim(t.uf)) = upper(trim(u.uf))
    where u.uf is null and t.uf <> ''
""")

In [ ]:
# g) UF repetida no cadastro JSON, com região em grafias diferentes? (DEFEITO 7)
q(f"""
    select uf, count(*) as vezes, string_agg(regiao, ' | ') as grafias
    from delta_scan('{bronze('ufs')}')
    group by 1
    having count(*) > 1
""")

## 4. Um resumo do que você encontrou

Anote aqui (em texto mesmo) os problemas e a decisão tomada para cada um.
Essa lista é o roteiro da Silver — e cada linha dela vira, mais adiante, um teste no `schema.yml`.

| # | Problema | Onde | Decisão |
|---|---|---|---|
| 1 | vírgula decimal na taxa | stg_taxas | `replace(',', '.')` antes de tipar |
| 2 | UF com grafia inconsistente | stg_taxas | `upper(trim(uf))` |
| 3 | UF órfã (`ZZ`) | fato_taxa | filtrar + teste `relationships` |
| 4 | taxa de abandono > 100 | fato_taxa | neutralizar na métrica derivada |
| 5 | código de município vazio | stg_taxas | descartar (sem chave) |
| 6 | duplicatas integrais | stg_taxas | `select distinct` |
| 7 | UF duplicada no JSON | stg_ufs | dedup por `row_number()` |

## 5. Depois do `dbt build`: Silver e Gold

Execute dentro da pasta `dbt/`:

```bash
dbt build
```

E então compare as camadas. A Silver e a Gold são **Parquet solto** (não Delta), então
voltamos a ler por caminho de arquivo.

In [ ]:
# Silver: dado tipado e limpo (repare nos TIPOS das colunas e na vírgula já corrigida)
q("select * from 'data/silver/stg_taxas.parquet' limit 10")

In [ ]:
# os tipos mudaram do Bronze (texto) para a Silver (números)?
bronze_tipos = q(f"select column_name as name, column_type as type from (describe select * from delta_scan('{bronze('taxas')}'))")
silver_tipos = q("select name, type from parquet_schema('data/silver/stg_taxas.parquet') where name <> 'schema'")
bronze_tipos.merge(silver_tipos, on="name", how="outer", suffixes=("_bronze", "_silver"))

In [ ]:
# Gold: o modelo dimensional. Repare na coluna abandono_combinado (a MÉTRICA DERIVADA)
q("select * from 'data/gold/fato_taxa.parquet' limit 10")

## 6. As perguntas de negócio

> **P1.** Qual é a taxa média de evasão (abandono) dos municípios, por REGIÃO e por UF?
>
> **P2.** A evasão é maior no FUNDAMENTAL ou no MÉDIO — e como varia por região?

A métrica `abandono_combinado` já existe na `fato_taxa` — quem consulta apenas agrupa.

**Resultado esperado (P1):** evasão média mais alta no **Nordeste** (~2,5%) e mais baixa
no **Centro-Oeste** (~0,9%).

In [ ]:
# P1 — evasão média por região e UF (o star schema: fato + dim_uf)
q("""
    select
        u.regiao,
        u.uf,
        count(*)                              as municipios,
        round(avg(f.abandono_combinado), 2)   as evasao_media_pct
    from 'data/gold/fato_taxa.parquet' f
    join 'data/gold/dim_uf.parquet'    u on f.uf_sk = u.uf_sk
    where f.abandono_combinado is not null
    group by 1, 2
    order by u.regiao, evasao_media_pct desc
""")

### 6.1 P2 e uma armadilha

Abaixo: a comparação Fundamental x Médio por região — e o erro que é fácil cometer com a taxa implausível.

In [ ]:
# P2 — a evasão é maior no Fundamental ou no Médio? (por região)
q("""
    select
        u.regiao,
        count(*)                              as municipios,
        round(avg(f.taxa_abandono_fund), 2)   as evasao_fundamental_pct,
        round(avg(f.taxa_abandono_med), 2)    as evasao_medio_pct,
        round(avg(f.taxa_abandono_med) - avg(f.taxa_abandono_fund), 2) as gap_medio_menos_fund
    from 'data/gold/fato_taxa.parquet' f
    join 'data/gold/dim_uf.parquet'    u on f.uf_sk = u.uf_sk
    where f.taxa_abandono_fund between 0 and 100
      and f.taxa_abandono_med  between 0 and 100
    group by 1
    order by evasao_medio_pct desc
""")

In [ ]:
# ⚠ A ARMADILHA: e se não filtrássemos a taxa implausível (> 100)?
# Compare a média do abandono do Fundamental COM e SEM o filtro de plausibilidade.
q("""
    select
        round(avg(taxa_abandono_fund), 3)                                          as media_sem_filtro,
        round(avg(case when taxa_abandono_fund between 0 and 100
                       then taxa_abandono_fund end), 3)                            as media_com_filtro
    from 'data/gold/fato_taxa.parquet'
""")
# Um único município com taxa 150 (impossível: passa de 100%) puxa a média para cima.
# Por isso a Gold NEUTRALIZA esse valor ao calcular abandono_combinado — a decisão
# mora no pipeline, uma vez, para todos (ver DECISOES.md, decisão 4).

---

**Fechando o raciocínio:** o mesmo dado apareceu quatro vezes neste notebook —
na fonte, no Bronze, na Silver e na Gold. O que mudou a cada passo?

- fonte → Bronze: o **formato** (CSV/JSON → Delta) e a preservação versionada
- Bronze → Silver: o **conteúdo** (qualidade: vírgula, grafia, duplicatas)
- Silver → Gold: a **organização** (modelo dimensional + a métrica derivada)
- Gold → consulta: o **significado** (evasão por região/UF — informação para decidir)